In [7]:
import pandas as pd

# Load original splits
train = pd.read_csv("../data/train.csv")
validation = pd.read_csv("../data/validation.csv")
test = pd.read_csv("../data/test.csv")

print("Before cleaning:")
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


# --------------------------------------------------
# 1. Remove exact duplicate rows within each split
# --------------------------------------------------

train = train.drop_duplicates()
validation = validation.drop_duplicates()
test = test.drop_duplicates()


# --------------------------------------------------
# 2. Remove messages having conflicting labels
# --------------------------------------------------

all_data = pd.concat([train, validation, test], ignore_index=True)

label_counts = (
    all_data.groupby("message")["target"]
    .nunique()
)

conflicting_messages = label_counts[label_counts > 1].index

print("Conflicting messages:", len(conflicting_messages))

train = train[~train["message"].isin(conflicting_messages)]
validation = validation[
    ~validation["message"].isin(conflicting_messages)
]
test = test[
    ~test["message"].isin(conflicting_messages)
]


# --------------------------------------------------
# 3. Remove cross-split duplicates
# --------------------------------------------------

train_messages = set(train["message"])

# Remove messages from validation/test if already present in train
validation = validation[
    ~validation["message"].isin(train_messages)
]

test = test[
    ~test["message"].isin(train_messages)
]


# Remove validation messages from test as well
validation_messages = set(validation["message"])

test = test[
    ~test["message"].isin(validation_messages)
]


# --------------------------------------------------
# 4. Reset indexes
# --------------------------------------------------

train = train.reset_index(drop=True)
validation = validation.reset_index(drop=True)
test = test.reset_index(drop=True)


# --------------------------------------------------
# Final dataset sizes
# --------------------------------------------------

print("\nAfter cleaning:")
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Before cleaning:
Train: (1722, 2)
Validation: (192, 2)
Test: (638, 2)
Conflicting messages: 2

After cleaning:
Train: (1659, 2)
Validation: (183, 2)
Test: (606, 2)


In [9]:
# Save cleaned datasets

train.to_csv("../data/train_clean.csv", index=False)
validation.to_csv("../data/validation_clean.csv", index=False)
test.to_csv("../data/test_clean.csv", index=False)

print("Clean datasets saved successfully!")

Clean datasets saved successfully!


In [10]:
print("Clean train:", pd.read_csv("../data/train_clean.csv").shape)
print("Clean validation:", pd.read_csv("../data/validation_clean.csv").shape)
print("Clean test:", pd.read_csv("../data/test_clean.csv").shape)

Clean train: (1659, 2)
Clean validation: (183, 2)
Clean test: (606, 2)
